<a href="https://colab.research.google.com/github/riza93n-hub/data-science-2026/blob/main/Pertemuan12_RIZA_250401020014.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aktivitas Hands On Pertemuan 12
**Nama Mahasiswa:** RIZA  
**NIM:** 250401020014  
**Kelas:** IF401

In [ ]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder

# =====================================================================
# LANGKAH 1: GENERATE & EKSPLORASI DATASET TRANSAKSI
# =====================================================================
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    items = np.random.choice(produk, n_item, replace=False).tolist()
    transaksi.append(items)

# Menyuntikkan pola: Roti sering dibeli bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('=== Eksplorasi Data Transaksi ===')
print('Contoh 3 transaksi pertama:', transaksi[:3])
print('Total transaksi:', len(transaksi))

# =====================================================================
# LANGKAH 2: ONE-HOT ENCODING TRANSAKSI
# =====================================================================
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print('\n=== 5 Baris Pertama Tabel One-Hot Encoding ===')
print(df.head())

=== Eksplorasi Data Transaksi ===
Contoh 3 transaksi pertama: [['Keju', 'Roti', 'Mentega', 'Kopi', 'Selai'], ['Roti', 'Kopi', 'Teh', 'Selai', 'Mentega'], ['Kopi', 'Susu', 'Teh']]
Total transaksi: 50

=== 5 Baris Pertama Tabel One-Hot Encoding ===
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [ ]:
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

# =====================================================================
# LANGKAH 3: CARI FREQUENT ITEMSET DENGAN APRIORI
# =====================================================================
print('=== Eksperimen Nilai min_support ===')
for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Menggunakan min_support = 0.1 untuk analisis selanjutnya
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)

print('\n=== 10 Frequent Itemset Teratas ===')
print(freq_items.head(10).to_string(index=False))

# =====================================================================
# LANGKAH 4: BENTUK & SARING ATURAN ASOSIASI
# =====================================================================
# Membentuk aturan dengan minimal keyakinan/confidence 0.5 (50%)
rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
# Menyaring aturan yang memiliki nilai Lift > 1 (korelasi positif)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print('\n=== 10 Aturan Asosiasi Terkuat (Diurutkan berdasarkan Lift) ===')
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10).to_string(index=False))

=== Eksperimen Nilai min_support ===
min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan

=== 10 Frequent Itemset Teratas ===
 support                itemsets
    0.52      frozenset({Selai})
    0.46        frozenset({Teh})
    0.42    frozenset({Mentega})
    0.36      frozenset({Telur})
    0.34       frozenset({Keju})
    0.32       frozenset({Gula})
    0.32       frozenset({Kopi})
    0.32       frozenset({Roti})
    0.32       frozenset({Susu})
    0.24 frozenset({Teh, Selai})

=== 10 Aturan Asosiasi Terkuat (Diurutkan berdasarkan Lift) ===
                antecedents          consequents  support  confidence     lift
     frozenset({Teh, Keju})   frozenset({Telur})     0.12    0.857143 2.380952
frozenset({Mentega, Selai})    frozenset({Kopi})     0.10    0.625000 1.953125
    frozenset({Roti, Gula})   frozenset({Selai})     0.10    1.000000 1.923077
        frozenset({Sereal}) frozenset({Mentega})     0.14    0.7777

### Analisis Aturan Asosiasi:
* **Aturan Terkuat:** Kombinasi aturan `frozenset({Teh, Keju}) -> frozenset({Telur})` mencetak nilai *Lift* tertinggi sebesar **2.38**, diikuti oleh `frozenset({Mentega, Selai}) -> frozenset({Kopi})` (**1.95**), dan `frozenset({Roti, Gula}) -> frozenset({Selai})` (**1.92**).
* **Logika Bisnis:** Pola aturan ini sangat masuk akal secara komersial. Munculnya aturan `frozenset({Roti, Gula}) -> frozenset({Selai})` dan `frozenset({Roti}) -> frozenset({Selai})` (Lift: **1.32**) menunjukkan validitas data di mana konsumen pembeli roti dan gula memiliki kecenderungan kuat untuk membeli selai sebagai pelengkap sarapan mereka. Nilai *Lift* $> 1$ membuktikan hubungan ini valid dan bukan kebetulan acak di kasir.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# =====================================================================
# LANGKAH 5: REKOMENDER DENGAN CONTENT-BASED FILTERING
# =====================================================================
katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy', 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

# =====================================================================
# LANGKAH 6: BANDINGKAN KEDUA PENDEKATAN (PRODUK TARGET: ROTI)
# =====================================================================
produk_target = 'Roti'

# Rekomendasi berdasarkan pola transaksi (Apriori)
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]

print('=== PERBANDINGAN STRATEGI REKOMENDASI ===')
print(f'Target Produk: {produk_target}\n')
print('1. Rekomendasi via Association Rules (Apriori):')
print(rules_terkait[['consequents', 'lift']].head(3).to_string(index=False))
print('\n2. Rekomendasi via Content-Based Filtering (Kategori):')
print(rekomendasi_serupa(produk_target))

=== PERBANDINGAN STRATEGI REKOMENDASI ===
Target Produk: Roti

1. Rekomendasi via Association Rules (Apriori):
       consequents     lift
frozenset({Selai}) 1.923077
frozenset({Selai}) 1.322115

2. Rekomendasi via Content-Based Filtering (Kategori):
['Selai', 'Sereal', 'Susu']


## Pembahasan Perbandingan Pendekatan

1. **Konsistensi Hasil:** Kedua pendekatan menunjukkan hasil yang **sebagian konsisten**. Untuk target produk `Roti`, baik *Association Rules* maupun *Content-Based Filtering* sama-sama merekomendasikan **Selai**.
   * *Association Rules* fokus pada hubungan transaksional belanja: `{Roti} -> {Selai}`.
   * *Content-Based Filtering* merekomendasikan `['Selai', 'Sereal', 'Susu']` karena ketiganya berada di rumpun kategori yang sama di katalog (dalam hal ini, komoditas *Bakery* / sarapan).
2. **Kapan Menggunakan Masing-masing Pendekatan?**
   * **Association Rules:** Sangat tepat untuk program promosi *cross-selling* (seperti paket *bundling* produk berbeda jenis) dan tata letak fisik barang di rak mini market.
   * **Content-Based Filtering:** Sangat optimal saat menangani produk baru (*cold start problem*) yang belum punya riwayat transaksi di kasir, cukup merekomendasikannya berdasarkan kesamaan kategori produk.
3. **Sistem Rekomendasi Gabungan (Hybrid):** Pendekatan terbaik adalah menggabungkannya. Toko bisa menggunakan *Content-Based* untuk menampilkan "Produk Serupa" di bagian bawah etalase produk, dan menyodorkan rekomendasi dari *Association Rules* sewaktu konsumen sudah menaruh barang di keranjang belanja menjelang *checkout*.

## Kesimpulan

* **Apa yang Dipelajari:** Mempraktikkan eksplorasi pola asosiasi menggunakan algoritma *Apriori* (Market Basket Analysis) untuk menyaring aturan penjualan, sekaligus membangun sistem penyaring berbasis kemiripan katalog (*Content-Based Filtering*) menggunakan *Cosine Similarity*.
* **Temuan Utama:** Hubungan sebab-akibat pembelian (*Association Rules*) tidak selalu searah dengan kemiripan jenis barang (*Content-Based*). Nilai *min_support* yang pas sangat krusial agar model tidak menghasilkan aturan sampah atau justru kehilangan pola bermakna, sedangkan kombinasi metode *Hybrid* adalah solusi terbaik bagi industri retail modern.
* **Keterbatasan & Pertanyaan:**
  * *Keterbatasan:* Eksperimen pencarian aturan asosiasi ini masih menggunakan basis data berskala kecil (50 transaksi sintetis) dengan kategori produk tunggal tanpa menghitung kuantitas belanja per item.
  * *Pertanyaan:* Bagaimana cara mengukur performa efisiensi komputasi algoritma *Apriori* saat dihadapkan pada jutaan transaksi kasir di dunia nyata agar proses pencarian aturan tidak membuat server lambat?